# Нейросеть для предсказания калорийности блюд

Задача - обучить нейросеть, которая будет предсказывать калорийность блюд по их фотографии и текстовому описанию ингредиентов.

In [ ]:
import zipfile
from functools import partial
from pathlib import Path

import pandas as pd

from scripts.utils import get_dataset_url, parse_ingredients


## Подготовка данных

Загружаем набор данных.

In [ ]:
DATASET_URL = "https://disk.yandex.ru/d/kz9g5msVqtahiw"

download_url = get_dataset_url(DATASET_URL)

!wget -O nutrition.zip -nc "$download_url"

Распаковываем датасет.

In [ ]:
DATA_DIR = Path.cwd() / "data"

if Path(DATA_DIR).exists():
    print("Датасет уже распакован.")
else:
    with zipfile.ZipFile("nutrition.zip") as zip_archive:
        zip_archive.extractall()
        print("Датасет распакован.")

Теперь посмотрим на состав датасета.

In [ ]:
images = DATA_DIR.rglob("rgb.png")
images_count = sum(1 for item in images if item.is_file())

print(f"Количество фотографий блюд: {images_count}")

In [ ]:
DISH_CSV = "dish.csv"
INGREDIENTS_CSV = "ingredients.csv"

dishes_df = pd.read_csv(DATA_DIR / DISH_CSV, index_col="dish_id")
ingredients_df = pd.read_csv(DATA_DIR / INGREDIENTS_CSV, index_col="id")

print(f"Записей в {DISH_CSV}        : {len(dishes_df)}")
print(f"Записей в {INGREDIENTS_CSV} : {len(ingredients_df)}")

Всего в датасете:
- 3490 фотографий блюд,
- 3262 записи с информацией о блюдах и их состав,
- 555 записей об ингредиентах.

In [ ]:
ingredients_df.head()

In [ ]:
dishes_df.head()

Ингредиенты закодированы идентификаторами, которые нужно заменить на названия продуктов. Убираем префикс; оставшееся число будет номером ингредиента в `ingredients.csv`.

In [ ]:
parse_ingredients_fn = partial(
    parse_ingredients, ingredients_map=ingredients_df["ingr"].to_dict()
)

dishes_df["ingredients"] = dishes_df["ingredients"].apply(parse_ingredients_fn)
dishes_df.head()